# Exploring User Journey embeddings

In [1]:
import sys

sys.executable

'/home/thomas/Python/virtualenvs/uj_embeddings/bin/python'

## Imports

In [2]:
import shap

/home/thomas/Python/virtualenvs/uj_embeddings/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import pairwise_distances

In [9]:
import sklearn

## Creating some sample data

## Defining some general use functions

In [8]:
embeddings.values # returns array of ROW arrays -> DataFrame 1st row becomes np.array[0]

array([[2., 4., 3., ..., 3., 4., 3.],
       [2., 4., 5., ..., 4., 3., 3.],
       [5., 4., 5., ..., 5., 5., 2.],
       ...,
       [2., 3., 5., ..., 6., 4., 3.],
       [5., 2., 2., ..., 5., 5., 1.],
       [2., 3., 1., ..., 3., 5., 1.]])

In [9]:
embeddings.values[0] # 1st row of data

array([2., 4., 3., 4., 4., 2., 3., 3., 4., 4., 3., 3., 3., 4., 4., 3., 4.,
       2., 2., 3., 3., 6., 3., 4., 3.])

In [10]:
emb_array = embeddings.values
np.mean(emb_array[0])

3.32

In [11]:
np.nanmean(emb_array[0:10], axis=0) # component-wise

array([3.3       , 4.1       , 4.5       , 4.2       , 4.2       ,
       4.5       , 4.3       , 4.2       , 2.8       , 3.6       ,
       2.9       , 2.8       , 4.44444444, 4.1       , 4.1       ,
       3.4       , 4.        , 2.8       , 3.1       , 3.        ,
       4.        , 3.        , 4.5       , 4.6       , 2.4       ])

In [12]:
np.nanmean(emb_array[0:10], axis=1) # observation-wise

array([3.32      , 3.52      , 3.88      , 3.88      , 3.4       ,
       4.16      , 3.4       , 3.32      , 4.20833333, 4.04      ])

In [13]:
# FILTRAR VALORES SAFRAS
embeddings[embeddings.A1.isin([0,1])]

,A1,A2,A3,A4,A5,C1,C2,C3,C4,C5,...,N1,N2,N3,N4,N5,O1,O2,O3,O4,O5
21,1.0,6.0,6.0,1.0,5.0,5.0,4.0,4.0,2.0,3.0,...,2.0,5.0,5.0,4.0,6.0,5.0,1,6.0,6.0,2.0
22,1.0,5.0,6.0,5.0,6.0,4.0,3.0,2.0,4.0,5.0,...,2.0,2.0,2.0,2.0,2.0,6.0,1,5.0,5.0,2.0
25,1.0,6.0,6.0,1.0,6.0,5.0,2.0,5.0,1.0,1.0,...,2.0,3.0,1.0,2.0,1.0,6.0,4,5.0,5.0,3.0
30,1.0,6.0,5.0,6.0,3.0,5.0,5.0,5.0,4.0,3.0,...,5.0,5.0,5.0,6.0,6.0,4.0,3,3.0,6.0,5.0
32,1.0,5.0,6.0,5.0,4.0,1.0,5.0,6.0,4.0,6.0,...,1.0,2.0,1.0,3.0,6.0,6.0,6,5.0,6.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2779,1.0,6.0,1.0,6.0,6.0,6.0,6.0,5.0,1.0,3.0,...,1.0,2.0,2.0,4.0,2.0,5.0,6,4.0,5.0,1.0
2780,1.0,6.0,4.0,6.0,3.0,5.0,5.0,5.0,2.0,5.0,...,2.0,2.0,5.0,4.0,2.0,5.0,1,6.0,6.0,1.0
2782,1.0,NaN,6.0,NaN,2.0,5.0,5.0,5.0,2.0,3.0,...,3.0,4.0,5.0,4.0,4.0,5.0,1,5.0,6.0,3.0
2783,1.0,5.0,6.0,5.0,6.0,5.0,4.0,5.0,2.0,3.0,...,3.0,4.0,5.0,4.0,4.0,5.0,1,5.0,6.0,3.0


In [83]:
embeddings.head()

,A1,A2,A3,A4,A5,C1,C2,C3,C4,C5,...,N2,N3,N4,N5,O1,O2,O3,O4,O5,gender
0,2.0,4.0,3.0,4.0,4.0,2.0,3.0,3.0,4.0,4.0,...,4.0,2.0,2.0,3.0,3.0,6,3.0,4.0,3.0,1
1,2.0,4.0,5.0,2.0,5.0,5.0,4.0,4.0,3.0,4.0,...,3.0,3.0,5.0,5.0,4.0,2,4.0,3.0,3.0,2
2,5.0,4.0,5.0,4.0,4.0,4.0,5.0,4.0,2.0,5.0,...,5.0,4.0,2.0,3.0,4.0,2,5.0,5.0,2.0,2
3,4.0,4.0,6.0,5.0,5.0,4.0,4.0,3.0,5.0,5.0,...,5.0,2.0,4.0,1.0,3.0,3,4.0,3.0,5.0,2
4,2.0,3.0,3.0,4.0,5.0,4.0,4.0,5.0,3.0,2.0,...,3.0,4.0,4.0,3.0,3.0,3,4.0,3.0,3.0,1


In [89]:
embeddings.A1.shift(-1)

0       2.0
1       5.0
2       4.0
3       2.0
4       6.0
       ... 
2795    2.0
2796    2.0
2797    5.0
2798    2.0
2799    NaN
Name: A1, Length: 2798, dtype: float64

In [85]:
embeddings['gender_shift'] = embeddings.gender.shift()

In [87]:
embeddings['gender_shift_2'] = embeddings.gender_shift.shift()

In [88]:
embeddings.head()

,A1,A2,A3,A4,A5,C1,C2,C3,C4,C5,...,N4,N5,O1,O2,O3,O4,O5,gender,gender_shift,gender_shift_2
0,2.0,4.0,3.0,4.0,4.0,2.0,3.0,3.0,4.0,4.0,...,2.0,3.0,3.0,6,3.0,4.0,3.0,1,None,None
1,2.0,4.0,5.0,2.0,5.0,5.0,4.0,4.0,3.0,4.0,...,5.0,5.0,4.0,2,4.0,3.0,3.0,2,1,None
2,5.0,4.0,5.0,4.0,4.0,4.0,5.0,4.0,2.0,5.0,...,2.0,3.0,4.0,2,5.0,5.0,2.0,2,2,1
3,4.0,4.0,6.0,5.0,5.0,4.0,4.0,3.0,5.0,5.0,...,4.0,1.0,3.0,3,4.0,3.0,5.0,2,2,2
4,2.0,3.0,3.0,4.0,5.0,4.0,4.0,5.0,3.0,2.0,...,4.0,3.0,3.0,3,4.0,3.0,3.0,1,2,2


## Actual test cases

- Análise exploratória inicial
  - Quais safras?
  - Qual público? Volumetrias?
  - Preenchimento
  - Correlação entre si/entre as vars já pré selecionadas do modelo
---
- Redução de dimensionalidade
  - Variável: média aritmética de todas as coordenadas do embedding (intra embedding)
  - Variável: média aritmética das coordenadas do embedding (intra embedding) após redução de dimensionalidade usando PCA
  - Variável: média aritmética das coordenadas do embedding (intra embedding) após redução de dimensionalidade usando t-SNE
  - Variável: média aritmética das coordenadas do embedding (intra embedding) após redução de dimensionalidade usando MDS
---
- Componentes como variáveis
  - Treinar modelo V1 acrescentando as componentes do embedding como novas features
    - Avaliar importancia das componentes relativo as demais vars
  - Treinar modelo usando as componentes como vars. explicativas e usar o score como variável no modelo

- Distâncias entre embeddings 
  - (Embedding Médio) Variável: distância (entre embeddings) entre o embedding observado e embedding que sumariza (média componente a componente) todos os da ref
  - (Default Referência) Variável: distância (entre embeddings) entre o embedding observado e embedding que sumariza (média componente a componente) casos de default
  - (Não Default Referência) Variável: distância (entre embeddings) entre o embedding observado e embedding que sumariza (média componente a componente) casos de não default
  - (Estabilidade de Jornada) Variável: diferença entre o embedding da ref atual vs. embedding da ref anterior



DEPENDE DOS PRODUTORES DOS DADOS
- Definição de embeddings de jornada referência: DEFINIR EMBEDDING DE JORNADA DE CONTRATAÇÃO DE CREDIÁRIO
  - Variável: distância do embedding observado até o embedding referência


PRA ENTENDER MELHOR
- Como cada embedding é gerado?
- O que significa cada valor? As colunas possuem alguma semantica?
- As componentes são normalizadas/padronizadas? Algum outro tipo relevante de tratamento é feito?
- Com que frequência são gerados?
- Que público abrange? (todos os correntistas? somente algum subgrupo?)
- Qual histórico possuímos?
- Como/qnd pretendem produtizar esses dados? (qual esteira? previsao?)
- É possível gerarmos embeddings representativos de jornadas específicas/pré definidas?

### - Média das componentes do embedding

In [14]:
embeddings.head(5)

,A1,A2,A3,A4,A5,C1,C2,C3,C4,C5,...,N1,N2,N3,N4,N5,O1,O2,O3,O4,O5
0,2.0,4.0,3.0,4.0,4.0,2.0,3.0,3.0,4.0,4.0,...,3.0,4.0,2.0,2.0,3.0,3.0,6,3.0,4.0,3.0
1,2.0,4.0,5.0,2.0,5.0,5.0,4.0,4.0,3.0,4.0,...,3.0,3.0,3.0,5.0,5.0,4.0,2,4.0,3.0,3.0
2,5.0,4.0,5.0,4.0,4.0,4.0,5.0,4.0,2.0,5.0,...,4.0,5.0,4.0,2.0,3.0,4.0,2,5.0,5.0,2.0
3,4.0,4.0,6.0,5.0,5.0,4.0,4.0,3.0,5.0,5.0,...,2.0,5.0,2.0,4.0,1.0,3.0,3,4.0,3.0,5.0
4,2.0,3.0,3.0,4.0,5.0,4.0,4.0,5.0,3.0,2.0,...,2.0,3.0,4.0,4.0,3.0,3.0,3,4.0,3.0,3.0


In [21]:
np.nanmean(emb_array, axis=1)

array([3.32, 3.52, 3.88, ..., 3.44, 3.84, 2.48])

In [15]:
embeddings['AVG_ALL_CPTS'] = np.nanmean(emb_array, axis=1)

### - Distâncias entre embeddings

> #### Embedding médio

In [16]:
emb_med = np.nanmean(emb_array, axis=0)

In [17]:
# from sklearn.metrics.pairwise import pairwise_distances
X = [[0, 0, 0], [1, 1, 1]]
Y = [[1, 0, 0], [1, 1, 0]]
pairwise_distances(X, Y, metric='sqeuclidean')

array([[1., 2.],
       [2., 1.]])

In [33]:
pairwise_distances(emb_array, emb_med.reshape(1,-1), metric='nan_euclidean')

array([[6.19567391],
       [6.10250965],
       [4.64634675],
       ...,
       [6.42887418],
       [7.45203829],
       [8.57451454]])

In [34]:
pairwise_distances(emb_array, emb_med.reshape(1,-1), metric='sqeuclidean', force_all_finite=False)

array([[38.38637515],
       [37.24062402],
       [21.58853816],
       ...,
       [41.33042316],
       [55.53287465],
       [73.52229953]])

In [31]:
import sklearn
sklearn.metrics.pairwise.PAIRWISE_DISTANCE_FUNCTIONS

{'cityblock': <function sklearn.metrics.pairwise.manhattan_distances(X, Y=None)>,
 'cosine': <function sklearn.metrics.pairwise.cosine_distances(X, Y=None)>,
 'euclidean': <function sklearn.metrics.pairwise.euclidean_distances(X, Y=None, *, Y_norm_squared=None, squared=False, X_norm_squared=None)>,
 'haversine': <function sklearn.metrics.pairwise.haversine_distances(X, Y=None)>,
 'l2': <function sklearn.metrics.pairwise.euclidean_distances(X, Y=None, *, Y_norm_squared=None, squared=False, X_norm_squared=None)>,
 'l1': <function sklearn.metrics.pairwise.manhattan_distances(X, Y=None)>,
 'manhattan': <function sklearn.metrics.pairwise.manhattan_distances(X, Y=None)>,
 'precomputed': None,
 'nan_euclidean': <function sklearn.metrics.pairwise.nan_euclidean_distances(X, Y=None, *, squared=False, missing_values=nan, copy=True)>}

In [ ]:
pairwise_distances(emb_array, emb_med.reshape(1,-1), metric='nan_euclidean', force_all_finite='allow-nan')

In [42]:
embeddings['DIST_AVGEMB_EUC'] = pairwise_distances(emb_array, emb_med.reshape(1,-1), metric='nan_euclidean', force_all_finite='allow-nan')

In [43]:
embeddings

,A1,A2,A3,A4,A5,C1,C2,C3,C4,C5,...,N3,N4,N5,O1,O2,O3,O4,O5,AVG_ALL_CPTS,DIST_AVGEMB_EUC
0,2.0,4.0,3.0,4.0,4.0,2.0,3.0,3.0,4.0,4.0,...,2.0,2.0,3.0,3.0,6,3.0,4.0,3.0,3.320000,6.195674
1,2.0,4.0,5.0,2.0,5.0,5.0,4.0,4.0,3.0,4.0,...,3.0,5.0,5.0,4.0,2,4.0,3.0,3.0,3.520000,6.102510
2,5.0,4.0,5.0,4.0,4.0,4.0,5.0,4.0,2.0,5.0,...,4.0,2.0,3.0,4.0,2,5.0,5.0,2.0,3.880000,4.646347
3,4.0,4.0,6.0,5.0,5.0,4.0,4.0,3.0,5.0,5.0,...,2.0,4.0,1.0,3.0,3,4.0,3.0,5.0,3.880000,6.590226
4,2.0,3.0,3.0,4.0,5.0,4.0,4.0,5.0,3.0,2.0,...,4.0,4.0,3.0,3.0,3,4.0,3.0,3.0,3.400000,4.799252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2795,6.0,1.0,3.0,3.0,3.0,6.0,6.0,6.0,1.0,1.0,...,1.0,NaN,1.0,6.0,1,6.0,6.0,1.0,3.375000,9.584104
2796,2.0,4.0,4.0,3.0,5.0,2.0,3.0,4.0,4.0,3.0,...,2.0,3.0,3.0,6.0,3,5.0,4.0,2.0,3.333333,4.972702
2797,2.0,3.0,5.0,2.0,5.0,5.0,5.0,5.0,1.0,1.0,...,3.0,3.0,1.0,5.0,1,6.0,4.0,3.0,3.440000,6.428874
2798,5.0,2.0,2.0,4.0,4.0,5.0,5.0,5.0,2.0,6.0,...,6.0,4.0,1.0,5.0,2,5.0,5.0,1.0,3.840000,7.452038


- O que ja foi feito
  - Cluster:
        - fit: 2301
        - score: 2302-2304
        - treino: 2302-2303
        - teste: 2304
  - Distâncias:
        - fit: 2301-2303
        - score: 2302-2304
        - treino: 2302-2303
        - teste: 2304
    - Média por componente dos emb. de clientes maus (default) ref anterior
    - Média por componente dos emb. de clientes bons ref anterior
    - Diferentes distâncias calculadas: euclidiana, euclidiana-nan, manhattan, coseno
---
- Ainda por fazer
  - Modelo usando coordenadas como variaveis explicativas
    - usar score do modelo como variável de teste
  - ç

In [ ]:
# ENTRADA: BASE COM PK_RIZNUMDOC_XX, REF (PK_DATREF_XX, SAFRA, SAFRA_MDL), agg, window

In [ ]:
exp_feature(
    src_database='rt2',
    src_table='tbdy278_pix_emdo',
    src_ref_field='pk_datref_xx',
    src_time_gran='d',
    out_database='workspace_db',
    out_table='tms_test_exp_feature',
    out_time_gran='m',
    out_s3_path='',
    aggregations=['SUM','AVG','MAX','MIN','COUNT'],
    feat_columns=['vlr_tran_finn'],
    agg_by=['pk_riznumdoc_xx'],
    time_windows=[1,3,6,12]    
)

In [ ]:
import uuid
import pandas as pd
import awswrangler as wr
from typing import Optional, Union

# PENSAR EM ADICIONAR PÚBLICO PARA JOIN

def exp_feature(
        src_database:Optional[str] = None,
        src_table:Union[str, pd.DataFrame],
        src_ref_field:Optional[str] = None,
        src_time_gran:str,                                     # 'd'|'w'|'m'
        out_database:str = 'workspace_db',
        out_table:Optional[str] = None,
        out_time_gran:str,                                     # 'd'|'w'|'m'
        out_s3_path:Optional[str] = None,
        aggregations:Optional[list[str]] = None                # ['SUM','AVG','MAX','MIN','COUNT'],
        feat_columns:[str],
        agg_by:Union[str, list[str]] = None,                   
        time_windows:Optional[list[int]] = None                # out_gran = 'd' -> [1,15,30,...] | 'w' -> [1,2,4...] | 'm' -> [1,3,6,...]
    ) -> Optional[pd.DataFrame]:


    if not out_s3_path:
        out_s3_path = f'{bucket}/exp_feature/temp/{uuid.uuid4().hex}/'

        
    # CASO A TABELA DE ORIGEM JÁ EXISTA NO CATÁLOGO DE DADOS
    if isinstance(src_table, str):
        # CRIAR NOVA TABELA REPARTICIONADA NO ATHENA/S3
        
        if not src_database:
            src_database = 'workspace_db'
            
        
        # VERIFICA SE A TABELA EXISTE
        table_exist = wr.catalog.does_table_exist(database=src_database, table=table)
        if not table_exist:
            raise ValueError(f"The table name provided does not exist/can not be found under '{src_database}'.'{src_table}'")
        else:
            location = wr.catalog.get_table_location(database=src_database, table=src_table)
            columns_types, partitions_types = wr.s3.read_parquet_metadata(path=location, dataset=True)
            table_columns = list(columns_types.keys()) + list(partition_keys.keys())

        
        if not src_ref_field:
            def_ref_fields = ['pk_datref_xx', 'safra', 'safra_mdl']
            for field in def_ref_fields:
                if field in table_columns:
                    src_ref_field = field
                    break
            else:
                raise ValueError(f"The 'src_ref_field' argument was not provided and no other likely option could be identified in your table!")


        # IDENTIFICAR GRANULARIDADE DA TABELA DE ORIGEM
        if src_time_gran:
            # VALIDA ENTRADA CASO VALOR SEJA FORNECIDO
            if src_time_gran not in ['d','w','m']:
                raise ValueError()
        else:
            # DO CONTRÁRIO, TENTA INFERIR A GRANULARIDADE ATRAVÉS DAS PARTIÇÕES
            if not partitions_types:
                raise ValueError(f"No 'src_time_gran' parameter has been provided and the table is not partitioned in a way that makes it possible to infer its time granularity! Please, provide the appropriate value for the 'src_time_gran' parameter.")
            
           partitions = wr.catalog.get_partitions(
                database='default',
                table='my_table'
            )
            parts = sorted(list(partitions_types.values()))
            anomesdia_patt = r'\d{8}'
            anomes_patt = r'\d{6}'
            # CONTINUAR
        
        # CHAMA QUERY CTAS QUE REPARTICIONA TABELA
        uuid_table = uuid.uuid4().hex
        wr.athena.create_ctas_table(
            sql=f"select {', '.join(agg_by + [src_ref_field] + feat_columns)} from {src_table}",
            database=src_database,
            ctas_database="workspace_db",
            storage_format="parquet",
            write_compression="snappy",
            partitioning_info=[src_ref_field],
            bucketing_info=(agg_by, 30),
            wait=True
        )
        
        # query1 = (f'''
        #     CREATE TABLE workspace_db.temp_exp_feature_{uuid_table}
        #     WITH (
        #         external_location = {s3_path},
        #         partiotioned_by = ARRAY[{ref_field}],
        #         bucketed_by = ARRAY['hash_mod_id'],
        #         bucket_count = 30,
        #         format = 'PARQUET'
        #     ) AS
        #     SELECT 
        # ''')
        
        pass
    elif isinstance(src_table, pd.DataFrame):
        # CRIA TABELA A PARTIR DO DATAFRAME JÁ COM AS DEVIDAS PARTIÇÕES
        pass

    
    # RODAR QUERIES DE AGREGAÇÃO FILTRANDO DEVIDAS PARTIÇÕES
    if 1 in time_windows:
        # RODA QUERY FILTRANDO SOMENTE ÚLTIMA REF
        pass
    elif 3 in time_windows:
        # RODA QUERY FILTRANDO 3 ÚLTIMAS REFS
        pass
    elif 6 in time_windows:
        # RODA QUERY FILTRANDO 6 ÚLTIMAS REFS
        pass
    elif 12 in time_windows:
        # RODA QUERY FILTRANDO 12 ÚLTIMAS REFS
        pass

In [ ]:
exp_feature(
    src_database='database_rt2',
    src_table='tbdy379_pix_emdo',
    id_field='pk_riznumdoc_xx',
    datref_field='pk_datref_xx',
    out_database='workspace_db',
    out_table_name='tms_test_exp_feature',
    out_s3_path='thommic/exp_feature/pix_emdo/',
    aggregations=['SUM','AVG','MAX','MIN','COUNT'],
    feat_columns=['vlr_tran_finn'],
    time_windows=[1,3,6,12]    
)

In [ ]:
import boto3
import uuid
import pandas as pd
import awswrangler as wr
from typing import Optional, Union

# PENSAR EM ADICIONAR PÚBLICO PARA JOIN


def exp_feature(
        src_database:Optional[str] = None,
        src_table:Union[str, pd.DataFrame],
        id_field:Optional[str] = None,
        datref_field:Optional[str] = None,
        out_database:Optional[str] = None,
        out_table_name:Optional[str] = None,
        out_s3_path:Optional[str] = None,
        aggregations:Optional[list[str]] = None,                # ['SUM','AVG','MAX','MIN','COUNT'],
        feat_columns:[str],                
        time_windows:Optional[list[int]] = None,                # out_gran = 'd' -> [1,15,30,...] | 'w' -> [1,2,4...] | 'm' -> [1,3,6,...]
    ) -> Optional[pd.DataFrame]:


    # DEFINE DATABSE ORIGEM CASO NÂO SEJA FORNECIDO
    if not src_database:
        src_database = 'workspace_db'

    if not aggregations:
        aggregations = ['SUM']
    else:
        def_aggregations = ['SUM','AVG','MAX','MIN','COUNT']
        for agg in aggregations:
            if agg not in def_aggregations:
                raise ValueError(f"The 'aggregations' parameter supports only the ['SUM','AVG','MAX','MIN','COUNT'] values. '{agg}' is not a supported aggregation value!")



    # CASO A TABELA DE ORIGEM JÁ EXISTA NO CATÁLOGO DE DADOS
    if isinstance(src_table, str):        
        # VERIFICA SE A TABELA EXISTE
        table_exist = wr.catalog.does_table_exist(database=src_database, table=src_table)
        if not table_exist:
            raise ValueError(f"The table name provided does not exist/can not be found under '{src_database}'.'{src_table}'")
        else:
            location = wr.catalog.get_table_location(database=src_database, table=src_table)
            columns_types, partitions_types = wr.s3.read_parquet_metadata(path=location, dataset=True)
            table_columns = list(columns_types.keys()) + list(partition_keys.keys())

        is_catalog = True
        
    elif isinstance(src_table, pd.DataFrame):
        table_columns = list(src_table.columns)
        is_catalog = False


    
        
    for feature in feat_columns:
        if feature not in table_columns:
            raise ValueError(f"The column '{feature}', passed as 'feat_columns' argument, could not be found in table '{src_database}'.'{src_table}'")
        
    if not id_field:
        def_id_fields = ['pk_riznumdoc_xx', 'cpf', 'cpf_cnpj', 'cod_chave_coro']
        for field in def_id_fields:
            if field in table_columns:
                id_field = field
                break
        else:
            raise ValueError(f"The 'id_field' argument was not provided and no other likely option could be identified in your table!")
    else:
        if id_field not in table_columns:
            raise ValueError(f"The column '{id_field}', passed as 'id_field' argument, could not be found in table '{src_database}'.'{src_table}'")

        
    if not datref_field:
        def_ref_fields = ['pk_datref_xx', 'safra', 'safra_mdl']
        for field in def_ref_fields:
            if field in table_columns:
                datref_field = field
                break
        else:
            raise ValueError(f"The 'datref_field' argument was not provided and no other likely option could be identified in your table!")
    else:
        if datref_field not in table_columns:
            raise ValueError(f"The column '{datref_field}', passed as 'datref_field' argument, could not be found in table '{src_database}'.'{src_table}'")


    # DEFINE BUCKET
    sts_client = boto3.client('sts')
    bucket = 'itau-self-wkp-sa-east-1-' + sts_client.get_caller_identity().get('Account')
    
    uuid_temp_location = uuid.uuid4().hex
    temp_location = f's3://{bucket}/exp_feature/temp/{uuid_temp_location}/'
    
    uuid_temp_table_name = str(uuid.uuid4().hex).replace('-','_')
    temp_table_name = f'exp_feature_temp_table_{}'
    
    temp_database = 'workspace_db'
    
    if is_catalog:
        # CHAMA QUERY CTAS QUE REPARTICIONA TABELA
        wr.athena.create_ctas_table(
            sql=f"select {', '.join([id_field] + feat_columns + datref_field)} from {src_table}",
            database=src_database,
            ctas_table=temp_table_name,
            ctas_database=temp_database,
            storage_format="parquet",
            write_compression="snappy",
            partitioning_info=[datref_field],
            bucketing_info=(id_field, 20),
            s3_output=temp_location,
            wait=True
        )
    else:
        # CRIA TABELA A PARTIR DO DATAFRAME JÁ COM AS DEVIDAS PARTIÇÕES
        wr.s3.to_parquet(
            df=src_table,
            database=temp_database,
            table=temp_table,
            compression='snappy'
            partition_cols=[datref_field],
            bucketing_info=([id_field], 20),
            path=temp_location,
            dataset=True           
        )

    
    if not out_database:
        out_database = 'workspace_db'

    if not out_table_name:
        uuid_table = uuid.uuid4().hex
        out_table_name = f'exp_feat_table_{uuid_table}'
        
    # DEFINE CAMINHO ONDE SALVAR A BASE DE SAÍDA
    if not out_s3_path:
        out_s3_path = f's3://{bucket}/exp_feature/temp/{uuid.uuid4().hex}/'
    elif out_s3_path.startswith('s3://'):
        out_s3_path = out_s3_path if out_s3_path[-1] == '/' else f'{out_s3_path}/'
    else:
        out_s3_path = f's3://{bucket}/{out_s3_path}'
        out_s3_path = out_s3_path if out_s3_path[-1] == '/' else f'{out_s3_path}/'




    # RODAR QUERIES DE AGREGAÇÃO FILTRANDO DEVIDAS PARTIÇÕES
    if 1 in time_windows:
        # RODA QUERY FILTRANDO SOMENTE ÚLTIMA REF
        pass
    elif 3 in time_windows:
        # RODA QUERY FILTRANDO 3 ÚLTIMAS REFS
        pass
    elif 6 in time_windows:
        # RODA QUERY FILTRANDO 6 ÚLTIMAS REFS
        pass
    elif 12 in time_windows:
        # RODA QUERY FILTRANDO 12 ÚLTIMAS REFS
        pass

In [ ]:
exp_feature(
    src_audience='workspace_db.tms_pub_mdl', # base com pk_riznumoc_xx e pk_datref_xx
    src_stg = 'database_rt2.tbdy379_pix_emdo',
    stg_id_ref_fields=['pk_riznumdoc_xx','pk_datref_xx'],
    out_table_name='tms_test_exp_feature',
    out_s3_path='thommic/exp_feature/pix_emdo/',
    stg_feat_columns=['vlr_tran_finn'],
    aggregations=['SUM','AVG','MAX','MIN','COUNT'],
    time_windows=['months',1,3,6,12]    
)

In [ ]:
import boto3
import uuid
import pandas as pd
import awswrangler as wr
from typing import Optional, Union

# PENSAR EM ADICIONAR PÚBLICO PARA JOIN


def exp_feature(
        src_audience:Union[str,pd.Dataframe],
        src_stg:str,
        stg_id_ref_fields:[str],
        out_table_name:str,
        out_s3_path:str,
        stg_feat_columns:[str],
        aggregations:[str],             
        time_windows:[str,int]
) -> Optional[pd.DataFrame]:

    # DEFINE BUCKET
    sts_client = boto3.client('sts')
    bucket = 'itau-self-wkp-sa-east-1-' + sts_client.get_caller_identity().get('Account')
    
    uuid_temp_location = uuid.uuid4().hex
    temp_location = f's3://{bucket}/exp_feature/temp/{uuid_temp_location}/'
    
    uuid_temp_table_name = str(uuid.uuid4().hex).replace('-','_')
    temp_table_name = f'exp_feature_temp_table_{}'
    
    temp_database = 'workspace_db'

    
    # CASO A TABELA DE ORIGEM JÁ EXISTA NO CATÁLOGO DE DADOS
    if isinstance(src_audience, str):        
        wr.athena.create_ctas_table(
            sql=f"select {', '.join(['pk_riznumdoc_xx'] + stg_feat_columns + ['pk_datref_xx'])} from {src_audience}",
            database=src_audience.split('.')[0],
            ctas_table=temp_table_name,
            ctas_database=temp_database,
            storage_format="parquet",
            write_compression="snappy",
            partitioning_info=['pk_datref_xx'],
            bucketing_info=('pk_riznumdoc_xx', 20),
            s3_output=temp_location,
            wait=True
        )
        
    elif isinstance(src_table, pd.DataFrame):
        wr.s3.to_parquet(
            df=src_table,
            database=temp_database,
            table=temp_table,
            compression='snappy'
            partition_cols=['pk_datref_xx'],
            bucketing_info=(['pk_riznumdoc_xx'], 20),
            path=temp_location,
            dataset=True           
        )






    
    if is_catalog:
        # CHAMA QUERY CTAS QUE REPARTICIONA TABELA

    else:
        # CRIA TABELA A PARTIR DO DATAFRAME JÁ COM AS DEVIDAS PARTIÇÕES


    
    if not out_database:
        out_database = 'workspace_db'

    if not out_table_name:
        uuid_table = uuid.uuid4().hex
        out_table_name = f'exp_feat_table_{uuid_table}'
        
    # DEFINE CAMINHO ONDE SALVAR A BASE DE SAÍDA
    if not out_s3_path:
        out_s3_path = f's3://{bucket}/exp_feature/temp/{uuid.uuid4().hex}/'
    elif out_s3_path.startswith('s3://'):
        out_s3_path = out_s3_path if out_s3_path[-1] == '/' else f'{out_s3_path}/'
    else:
        out_s3_path = f's3://{bucket}/{out_s3_path}'
        out_s3_path = out_s3_path if out_s3_path[-1] == '/' else f'{out_s3_path}/'




    # RODAR QUERIES DE AGREGAÇÃO FILTRANDO DEVIDAS PARTIÇÕES
    if 1 in time_windows:
        # RODA QUERY FILTRANDO SOMENTE ÚLTIMA REF
        pass
    elif 3 in time_windows:
        # RODA QUERY FILTRANDO 3 ÚLTIMAS REFS
        pass
    elif 6 in time_windows:
        # RODA QUERY FILTRANDO 6 ÚLTIMAS REFS
        pass
    elif 12 in time_windows:
        # RODA QUERY FILTRANDO 12 ÚLTIMAS REFS
        pass

- PROBLEMA
  Criar variáveis que traduzam a dinâmica de transações pix PF-PJ e avaliar sua capacidade de discriminar risco de crédito

- METODOLOGIA
  - Definir clusters de CNPJs
    - Quantos clusters definir?
    - Quais variáveis utilizar na geração dos clusters?
    - Como agregar os valores de transações da melhor maneira?
    - Como incorporar o tempo como dimensão do problema? Ou será que nem faz sentido?
      
  - Mapear dinâmica das transações PIX dos PFs que transacionam com cada cluster


  - Traduzir em variáveis explicativas para treinamento do modelo

- Finalizar cálculo dos centroides de cluster: média (por cluster) da quantidade e valor total de transações (por CPF) nas janelas de tempo
- Calcular vars pix dos CPFs do público: quantidade e valor total de transações (por CPF) nas janelas de tempo
- Calcular distancia de cada CPF a cada centroide usando as vars pix como coordenadas
- Usar as distancias como vars explicativas de modelagem

- Calcular centroide de um cluster identificando cnpjs de bets
- Calcular distancias dos cpfs do modelo ao cluster bet
- Distancia media para target 1 e 0
- Dividir distancias em quantis e calcular bad

In [1]:
import io
import os
import re
import shutil
import string
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.layers import TextVectorization

2024-05-28 02:05:30.302423: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-05-28 02:05:30.307045: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-05-28 02:05:30.365885: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-28 02:05:31.774384: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

dataset = tf.keras.utils.get_file("aclImdb_v1.tar.gz", url,
                                  untar=True, cache_dir='.',
                                  cache_subdir='')

dataset_dir = os.path.join(os.path.dirname(dataset), 'aclImdb')
os.listdir(dataset_dir)

84125825/84125825 ━━━━━━━━━━━━━━━━━━━━ 87s 1us/step


['imdbEr.txt', 'test', 'train', 'README', 'imdb.vocab']

In [3]:
train_dir = os.path.join(dataset_dir, 'train')
os.listdir(train_dir)

['unsup',
 'labeledBow.feat',
 'urls_unsup.txt',
 'pos',
 'neg',
 'urls_neg.txt',
 'unsupBow.feat',
 'urls_pos.txt']

In [4]:
remove_dir = os.path.join(train_dir, 'unsup')
shutil.rmtree(remove_dir)

In [5]:
batch_size = 1024
seed = 123
train_ds = tf.keras.utils.text_dataset_from_directory(
    'aclImdb/train', batch_size=batch_size, validation_split=0.2,
    subset='training', seed=seed)
val_ds = tf.keras.utils.text_dataset_from_directory(
    'aclImdb/train', batch_size=batch_size, validation_split=0.2,
    subset='validation', seed=seed)

Found 25000 files belonging to 2 classes.
Using 20000 files for training.
Found 25000 files belonging to 2 classes.
Using 5000 files for validation.


In [6]:
for text_batch, label_batch in train_ds.take(1):
  for i in range(5):
    print(label_batch[i].numpy(), text_batch.numpy()[i])

0 b"Oh My God! Please, for the love of all that is holy, Do Not Watch This Movie! It it 82 minutes of my life I will never get back. Sure, I could have stopped watching half way through. But I thought it might get better. It Didn't. Anyone who actually enjoyed this movie is one seriously sick and twisted individual. No wonder us Australians/New Zealanders have a terrible reputation when it comes to making movies. Everything about this movie is horrible, from the acting to the editing. I don't even normally write reviews on here, but in this case I'll make an exception. I only wish someone had of warned me before I hired this catastrophe"
1 b'This movie is SOOOO funny!!! The acting is WONDERFUL, the Ramones are sexy, the jokes are subtle, and the plot is just what every high schooler dreams of doing to his/her school. I absolutely loved the soundtrack as well as the carefully placed cynicism. If you like monty python, You will love this film. This movie is a tad bit "grease"esk (without

2024-05-28 02:10:50.890839: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [7]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [8]:
# Embed a 1,000 word vocabulary into 5 dimensions.
embedding_layer = tf.keras.layers.Embedding(1000, 5)

In [9]:
result = embedding_layer(tf.constant([1, 2, 3]))
result.numpy()

array([[-0.01176568, -0.00951239, -0.04114689, -0.04893839,  0.0435149 ],
       [-0.0324956 ,  0.00789318,  0.04283575, -0.01955197,  0.02206867],
       [-0.02693304, -0.01668467, -0.00042   ,  0.03686053, -0.01228829]],
      dtype=float32)

In [10]:
result = embedding_layer(tf.constant([[0, 1, 2], [3, 4, 5]]))
result.shape

TensorShape([2, 3, 5])

In [11]:
result.numpy()

array([[[-0.03121332, -0.02761109, -0.03741707, -0.0198403 ,
          0.04107579],
        [-0.01176568, -0.00951239, -0.04114689, -0.04893839,
          0.0435149 ],
        [-0.0324956 ,  0.00789318,  0.04283575, -0.01955197,
          0.02206867]],

       [[-0.02693304, -0.01668467, -0.00042   ,  0.03686053,
         -0.01228829],
        [ 0.04132598,  0.046374  , -0.02058183, -0.00618381,
         -0.03252601],
        [-0.0278337 , -0.0486428 , -0.03429434,  0.02602471,
         -0.02762508]]], dtype=float32)

In [13]:
# Create a custom standardization function to strip HTML break tags '<br />'.
def custom_standardization(input_data):
  lowercase = tf.strings.lower(input_data)
  stripped_html = tf.strings.regex_replace(lowercase, '<br />', ' ')
  return tf.strings.regex_replace(stripped_html,
                                  '[%s]' % re.escape(string.punctuation), '')


# Vocabulary size and number of words in a sequence.
vocab_size = 10000
sequence_length = 100

In [14]:
# Use the text vectorization layer to normalize, split, and map strings to
# integers. Note that the layer uses the custom standardization defined above.
# Set maximum_sequence length as all samples are not of the same length.
vectorize_layer = TextVectorization(
    standardize=custom_standardization,
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=sequence_length)

In [15]:
# Make a text-only dataset (no labels) and call adapt to build the vocabulary.
text_ds = train_ds.map(lambda x, y: x)

In [16]:
vectorize_layer.adapt(text_ds)

2024-05-28 02:14:55.382019: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [17]:
embedding_dim=16

model = Sequential([
  vectorize_layer,
  Embedding(vocab_size, embedding_dim, name="embedding"),
  GlobalAveragePooling1D(),
  Dense(16, activation='relu'),
  Dense(1)
])

In [18]:
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir="logs")

In [19]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [20]:
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[tensorboard_callback])

Epoch 1/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - accuracy: 0.5037 - loss: 0.6919 - val_accuracy: 0.4886 - val_loss: 0.6854
Epoch 2/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - accuracy: 0.5037 - loss: 0.6822 - val_accuracy: 0.4886 - val_loss: 0.6712
Epoch 3/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - accuracy: 0.5037 - loss: 0.6657 - val_accuracy: 0.4886 - val_loss: 0.6496
Epoch 4/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - accuracy: 0.5051 - loss: 0.6410 - val_accuracy: 0.4988 - val_loss: 0.6206
Epoch 5/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - accuracy: 0.5433 - loss: 0.6085 - val_accuracy: 0.5930 - val_loss: 0.5864
Epoch 6/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - accuracy: 0.6401 - loss: 0.5706 - val_accuracy: 0.6596 - val_loss: 0.5506
Epoch 7/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - accuracy: 0.7123 - loss: 0.5313 - val_accuracy: 0.7102 - val_loss: 0.5168
Epoch 8/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - accuracy: 0.7569 - loss: 0.4941 - val_accuracy: 0.7400 - v

In [21]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 100, 16)        │       160,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 480,869 (1.83 MB)

 Trainable params: 160,289 (626.13 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 320,580 (1.22 MB)

In [22]:
weights = model.get_layer('embedding').get_weights()[0]
vocab = vectorize_layer.get_vocabulary()

In [24]:
weights.shape

(10000, 16)

In [ ]:
out_v = io.open('vectors.tsv', 'w', encoding='utf-8')
out_m = io.open('metadata.tsv', 'w', encoding='utf-8')

for index, word in enumerate(vocab):
  if index == 0:
    continue  # skip 0, it's padding.
  vec = weights[index]
  out_v.write('\t'.join([str(x) for x in vec]) + "\n")
  out_m.write(word + "\n")
out_v.close()
out_m.close()

- Join tabela especs com pj bureau
- Avalia tempo desde fundacao da emp ate credenciamento no microcred
      - Caso seja pequeno (1o contrato de microcred) EXP_PREVIA 0
      - Caso seja longo
        - Ramo de atividade anterior era microcredito EXP_PREVIA 1
        - Ramo de atividade era diferente EXP_PREVIA 0

In [ ]:
CREATE TABLE workspace_db.tms_publico_microcred_espec_pjbureau
with (format = "Parquet", 
      write_compression = 'SNAPPY',
        external_location = 's3://itau-self-wkp-sa-east-1-635985251289/thommic/microcred_espec/bas')
as with contratos as (
SELECT 
tempo desde fundacao ate cadastro
SELECT
I
)
A.*,
SUBSTRING(CAST(A.Data_Cred as VARCHAR), 1,4) || SUBSTRING (CAST(A.Data_Cred as VARCHAR) as anomes_cred,
B.dat_abet_emp,
B.qtd_func
B.ind_micr_indl
    
FROM workspace_db.tms_publico_microcred_espec as A
LEFT JOIN db_corp_modelagem_datacredriskdevops_sot_01.tbdc6134_cada_pjur_brau as B
ON A.CNPJ8_Espec = B.num_raiz_docm_idef_pess_juri AND A.safa = CAST(B.num_Dat_rfrc_arqu as INT)
AND TRIM(B.ind_esto_ssto) = 'N' 
AND TRIM(B.ind_tipo_pess) = 'J' נ
SELECT B.
CAST(B.qtd_cntrts_app as DOUBLE)/qtd_contratos as pct_cntrt_app CAST(B.qtd_cntrt_reneg as DOUBLE)/qtd_contratos as pct_cntrts_reneg FROM contratos as B;





as with anterior_cred as(
    SELECT 
            CAST(B.num_raiz_docm_idef_pess_juri as bigint) as CNPJ8_Espec,
            B.dat_abet_emp,
            B.qtd_func,
            B.ind_micr_indl,
            MAX(B.num_dat_ref_arqu)
    FROM db_corp_modelagem_datacredriskdevops_sot_01.tbdc6134_cada_pjur_brau as B
    WHERE EXISTS (SELECT 1
                    FROM workspace_db.tms_publico_microcred_espec as A
                    WHERE A.CNPJ8_Espec = B.num_raiz_docm_idef_pess_juri
                            AND B.num_dat_ref_arqu < SUBSTRING(CAST(A.Data_Cred as VARCHAR), 1,4) || SUBSTRING (CAST(A.Data_Cred as VARCHAR))
                 )
    GROUP BY 1,2,3,4
)
SELECT 
        *
FROM 

LEFT JOIN anterior_cred 
ON A.CNPJ8_Espec = B.num_raiz_docm_idef_pess_juri AND A.safa = CAST(B.num_Dat_rfrc_arqu as INT)
AND TRIM(B.ind_esto_ssto) = 'N' 
AND TRIM(B.ind_tipo_pess) = 'J'
AND B.num_dat_ref_arqu < SUBSTRING(CAST(A.Data_Cred as VARCHAR), 1,4) || SUBSTRING (CAST(A.Data_Cred as VARCHAR)








as with antes_credenc as(
    SELECT B.*
           ,CAST(B.num_raiz_docm_idef_pess_juri as bigint) as CNPJ8_Espec
           ,CAST(B.num_dat_rfrc_arqu as int) as anomes
    FROM db_corp_modelagem_datacredriskdevops_sot_01.tbdc6134_cada_pjur_brau as B
    WHERE EXISTS (SELECT 1
                    FROM workspace_db.tms_publico_microcred_espec as A
                    WHERE A.CNPJ8_Espec = B.num_raiz_docm_idef_pess_juri
                            AND B.num_dat_ref_arqu < SUBSTRING(CAST(A.Data_Cred as VARCHAR), 1,4) || SUBSTRING (CAST(A.Data_Cred as VARCHAR))
                 ))
), ordem_cnpj as(
    ROW_NUMBER() OVER (PARTITION BY num_raiz_docm_idef_pess_juri ORDER BY num_dat_rfrc)

)




CPF espec e cruzar com base de visao socio
ordenar cnpjs atribuidos ao cpf em ordem decrescente limitando historico maximo a data de credenciamento no microcredito
e identificar o atual do microcredito

In [ ]:
lista_cred = ['CTRT_BC_NEGADO', 'CTRT_BC_SEM_PA', 'CTRT_CREDIARIO', 'INF_NEGADOS_CR', 'INF_SEM_PA_CRE']
lista_corr = ['CTRT_BC_NEGADO', 'CTRT_CREDIARIO', 'INF_NEGADOS_CR']
lista_ia = ['IA']
lista_iu = ['IU']
lista_ip = ['IP']
lista_iti = ['CTRT_BC_ITI']
lista_carton = ['CTRT_CP', 'CTRT_BC_CARTON']

def gini(*args):
    return (roc_auc_score(*args) - 0.5) / 0.5


def calc_gini_quebras (df, var_teste, var_finalistas, 1gbm, position):
df['FILTRO_IDADE'] = df['qt_ida_01m_bf'].apply(lambda x: 'Menor_25' if x < 25 else 'De_25_a_30' if 25 <= x <= 30
                                                                              else 'De_31_a_35' if 31 <= x <= 35
                                                                              else 'Acima_35')

df_teste_corr = df[(df[safra].isin(lista_safras_teste)) & df['origem').isin(lista_corr)]
df_teste_ia = df[(df[safra].isin(lista_safras_teste)) & (df['origem'].isin(lista_cred)) & (df['segmento'].isin(lista_ia)
df_teste_iu = df[(df[safra].isin(lista_safras_teste)) & (df['origem'].isin(lista_cred)) & (df['segmento'].isin(lista_iu)) 
df_teste_ip = df[(df[safra].isin(lista_safras_teste)) & (df['origem'].isin(lista_cred)) & (df['segmento').isin(lista_ip))
df_teste_carton = df[(df[safra].isin(lista_safras_teste)) & df['origem'].isin(lista_carton)] 
df_teste_iti = df[(df[safra].isin(lista_safras_teste)) & df['origem'].isin(lista_iti)]

df_menor_25 = df[(df[safra].isin(lista_safras_teste )) & (df['FILTRO_IDADE'] == 'Menor_25')] 
df_de_25_a_30 = df[(df [safra].isin(lista_safras_teste)) & (df['FILTRO_IDADE'] == 'De_25_a_30')]
df_de_31_a_35 = df[(df[safra].isin(lista_safras_teste)) & (df['FILTRO_IDADE'] == 'De_31_a_35')]
df_acima_35 = df[(df[safra].isin(lista_safras_teste)) & (df['FILTRO_IDADE'] == 'Acima_35')]

           
X_test_corr = df_teste_corr[var_finalistas]
y_test_corr = df_teste_corr[target]
x_test_ia = df_teste_ia[var_finalistas]
y_test_ia = df_teste_ia[target]





In [ ]:
コロC
Code
git
Embedding_clusters_pj.ipynb
X
df_de_25_a_30 = df[(df[safra].isin(lista_safras_teste)) & (df['FILTRO_IDADE'] == 'De_25_a_30')] df_de_31_a_35 = df[(df[safra].isin(lista_safras_teste)) & (df['FILTRO_IDADE'] == 'De_31_a_35')]
df_acima_35 = df[(df[safra].isin(lista_safras_teste)) & (df['FILTRO_IDADE'] == 'Acima_35')]
X_test_corr = df_teste_corr[var_finalistas]
y_test_corr = df_teste_corr[target]

x_test_ia = df_teste_ia[var_finalistas] 
y_test_ia = df_teste_ia[target]

x_test_iu = df_teste_iu[var_finalistas] 
y_test_iu = df_teste_iu[target]

x_test_ip = df_teste_ip[var_finalistas] 
y_test_ip = df_teste_ip[target]

X_test_carton = df_teste_carton[var_finalistas] 
y_test_carton = df_teste_carton[target]

]X_test_iti = df_teste_iti[var_finalistas] 
y_test_iti = df_teste_iti[target]

X_test_menor25 = df_menor_25[var_finalistas]
y_test_menor25 = df_menor_25[target]

X_test_25_a_30 = df_de_25_a_30[var_finalistas] 
y_test_25_a_30 = df_de_25_a_30[target]

X_test_31_a_35 = df_de_31_a_35[var_finalistas]
y_test_31_a_35 = df_de_31_a_35[target]

X_test_acima_35 = df_acima_35[var_finalistas] 
y_test_acima_35 = df_acima_35[target]

# Gini da base de teste out of time print('\nGINI nos subgrupos:')
# Gini da base de teste out of time
#print("PUBLICO FULL:")
df_test['SCORE'] = 1gbm.predict_proba(df_test[var_finalistas])[:,1]
y_pred_train = 1gbm.predict_proba(X_train[var_finalistas], raw_score=True) 
print("\nGINI TRAIN:", round(2 roc_auc_score(y_train, y_pred_train) 1, 3)) gini_train = round(2* roc_auc_score(y_train, y_pred_train) - 1, 3)

y_pred_val = 1gbm.predict_proba(X_val[var_finalistas], raw_score=True) 
print "GTNT VAI:". round 12 or aur scorelv val. v pred val) - 1, 3))

In [ ]:

y_test_carton = df_teste_carton[target]
X_test_iti = df_teste_iti[var_finalistas]
y_test_iti = df_teste_iti[target]
X_test_menor25 = df_menor_25[var_finalistas]
y_test_menor25 = df_menor_25[target]
X_test_25_a_30 = df_de_25_a_30[var_finalistas] y_test_25_a_30 = df_de_25_a_30[target]
X_test_31_a_35 = df_de_31_a_35[var_finalistas] 
aws:sagemaker:sa-east-1:635985251289:image/mlees-ubun...
No kernel: Unknown Insta